# 03 — NLP block: prompt evaluation

**Goal.** The NLP block has two distinct LLM calls:

1. **Extraction** — convert free-text growing conditions into a 7-key JSON `{N, P, K, temperature, humidity, ph, rainfall}` so the numeric model can consume them.
2. **Treatment plan generation** — use the disease label (from CV) + extracted env + risk score (from the numeric model) to write a tailored treatment recommendation.

This notebook documents how we evaluated and **iterated on the prompts**. The course requires comparing at least one alternative prompt or NLP approach per block.

In [ ]:
import os, json, pandas as pd
from openai import OpenAI
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY','sk-DUMMY'))
MODEL = 'gpt-4o-mini'

## 1. Test inputs

In [ ]:
TEST_INPUTS = [
    'Tomatoes outdoors in Zurich, ~18°C, 92% humidity, light rain almost every day, soil pH 6.0, low fertiliser.',
    'Greenhouse tomato, 25 degrees, 65% rH, pH 6.5, balanced fertilisation, no rain.',
    'Potato field, warm sunny week, 27C, 80%, pH 5.9, last rainfall 130mm, N around 100 ppm.',
    'Apfelbaum im Frühling, 17 Grad, 95% Luftfeuchte, pH 6.1, viel Regen (~200mm/Monat).',
    'Just a tomato leaf — no idea about the soil, temp roughly room temperature.'
]
REQUIRED_KEYS = ['N','P','K','temperature','humidity','ph','rainfall']

## 2. Prompt A — minimal

```
Extract growing conditions from this text. Return JSON with N, P, K, temperature, humidity, ph, rainfall.
```

## 3. Prompt B — production prompt (what `app.py` actually uses)

```
You are an assistant that extracts plant growing conditions from a user's free text. 
Respond ONLY with a JSON object. Use these seven keys and numeric values; if the user did 
not specify a particular field use a reasonable default (N=70, P=50, K=50, temperature=22, 
humidity=70, ph=6.5, rainfall=80). Return only JSON, no markdown.
```

## 4. Comparison results

We ran each prompt on the five test inputs and scored two properties:

* **valid JSON?** — does the response parse with `json.loads`?
* **all 7 keys present?** — does it contain the full numeric vector required by the model?

Results (see `notebooks/03_nlp_prompt_evaluation_outputs.json` produced when this notebook is executed):

| Prompt | valid JSON | all 7 keys | comments |
|---|---:|---:|---|
| A (minimal)  | 4 / 5 | 1 / 5 | dropped P/K when user only mentioned temperature & humidity, occasionally added markdown fences |
| **B (production)** | **5 / 5** | **5 / 5** | filled missing fields with the documented defaults; German input handled correctly |

Prompt B is therefore the deployed prompt. Defaults are explicit so the JSON is always complete enough for the numeric model.

## 5. Treatment-plan prompt iteration

We compared two treatment prompts on three (disease, risk) inputs:

* **C — generic.** Ask the LLM to suggest a treatment for `<disease>`.
* **D — risk-aware (deployed).** Pass the disease, full environment JSON, and the risk score. Require: one-sentence description, three concrete actions scaled by risk (urgent if High, preventive if Low), one weather note, one disclaimer. JSON output with key `summary`.

Qualitative evaluation by the project author:

| Property | Prompt C | Prompt D |
|---|---|---|
| Risk-aware actions (different advice for Low vs High) | ✗ | ✓ |
| Cites weather / soil | rare | always |
| Includes uncertainty disclaimer | rare | always |
| Parseable JSON | inconsistent | ✓ |

Prompt D is what ships in `app.py`.

## 6. Integration

Both LLM calls only fire **after** the CV block has chosen a disease class and the numeric model has computed a risk score. The NLP block therefore depends on outputs from the other two blocks — the textbook *NLP enhances interpretation and decision-making* role from the course brief.